In [3]:
# ============================================================
# Cell 0: install dependencies if needed
# ============================================================

# In Jupyter, uncomment if needed:
# !uv pip install -U trl datasets accelerate z3-solver transformers safetensors

In [1]:
# ============================================================
# Cell 1: imports and configuration
# ============================================================

import os
import re
import ast
import json
import random
import inspect
from pathlib import Path
from typing import Any, Dict, List, Optional

import torch
from datasets import Dataset
from z3 import Bool, Or, Not, Solver, sat, unsat
from transformers import AutoTokenizer

from trl import GRPOTrainer, GRPOConfig


# ----------------------------
# Paths
# ----------------------------

REPO_ROOT = Path(".").resolve()

# Start RL from your SFT model, not the base model.
MODEL_DIR = Path("results/sft_qwen35_08b_base_sat_unsat_full/final_model")

# Unused teacher records.
RECORD_DIR = Path("teacher_responses_qwen35_2b/records")

# Save small GRPO test outputs here.
OUTPUT_DIR = Path("results/grpo_z3_qwen35_08b_test")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ----------------------------
# Small notebook-test settings
# ----------------------------

SEED = 42

# Start tiny. Increase after the pipeline works.
N_PER_LABEL = 8          # 8 SAT + 8 UNSAT for notebook smoke test
MAX_STEPS = 5            # tiny GRPO test
NUM_GENERATIONS = 2      # GRPO needs >1 generation per prompt

MAX_PROMPT_LENGTH = 4096
MAX_COMPLETION_LENGTH = 512

random.seed(SEED)
torch.manual_seed(SEED)

print("Repo root:", REPO_ROOT)
print("Model dir:", MODEL_DIR, "exists:", MODEL_DIR.exists())
print("Record dir:", RECORD_DIR, "exists:", RECORD_DIR.exists())
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

Repo root: /scratch/network/yd1202/COS598B-project
Model dir: results/sft_qwen35_08b_base_sat_unsat_full/final_model exists: True
Record dir: teacher_responses_qwen35_2b/records exists: True
CUDA available: True
GPU: NVIDIA A100 80GB PCIe
bf16 supported: True


In [5]:
# ============================================================
# Cell 2: helpers to load teacher records and build no-leak prompts
# ============================================================

def load_json(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def extract_tag(text: str, tag: str) -> Optional[str]:
    m = re.search(
        rf"<{re.escape(tag)}>\s*(.*?)\s*</{re.escape(tag)}>",
        text,
        flags=re.DOTALL | re.IGNORECASE,
    )
    return m.group(1).strip() if m else None


def parse_tag_json(text: str, tag: str):
    value = extract_tag(text, tag)
    if value is None:
        return None
    try:
        return json.loads(value)
    except Exception:
        return ast.literal_eval(value)


SAFE_SYSTEM_PROMPT = """You are a logical reasoning assistant solving SATBench-style natural-language logic puzzles.

Important rules:
- Use only the constraints stated in the <conditions> section.
- The <scenario> section is background only and adds no hidden constraints.
- All variables are independent Boolean decisions unless the conditions explicitly say otherwise.
- Do not add commonsense assumptions such as mutual exclusivity, exactly-one constraints, or real-world causal links.
- Variables not mentioned in the conditions are irrelevant to satisfiability.

Required output format:

<think>
Write concise reasoning here.
</think>

Decision: SAT or UNSAT

Certificate:
- If SAT, write exactly:
  Assignment: <nested 0/1 list matching dims>

  The assignment must contain exactly num_vars values in row-major order according to dims.
  Use 1 for True and 0 for False.
  Do not use ellipses, omitted entries, variable names, or prose inside the assignment list.

- If UNSAT, write exactly:
  UNSAT core condition numbers: <list of 1-indexed condition numbers>

  The condition numbers must refer to the numbered conditions in the <conditions> section.
  The list does not need to be minimal, but the selected conditions must be jointly unsatisfiable.
  Do not use 0-indexed clause indices.

Explanation:
Write a short explanation connecting the certificate to the conditions.

End with exactly one final label on its own line:
[SAT]
or
[UNSAT]

Do not write anything after the final label.

Example 1: SAT output format

Input facts:
<dims>
[2]
</dims>

<num_vars>
2
</num_vars>

<conditions>
1. Alice joins the club.
2. Bob does not join the club.
</conditions>

Correct output:
<think>
Condition 1 requires x(0) to be true. Condition 2 requires x(1) to be false. These requirements do not conflict, so the puzzle is satisfiable.
</think>

Decision: SAT

Certificate:
Assignment: [1, 0]

Explanation:
The assignment sets Alice to True and Bob to False, satisfying both conditions.

[SAT]

Example 2: UNSAT output format

Input facts:
<dims>
[2]
</dims>

<num_vars>
2
</num_vars>

<conditions>
1. Alice joins the club.
2. Alice does not join the club.
</conditions>

Correct output:
<think>
Condition 1 requires x(0) to be true, while condition 2 requires x(0) to be false. The same variable cannot be both true and false, so these conditions are jointly unsatisfiable.
</think>

Decision: UNSAT

Certificate:
UNSAT core condition numbers: [1, 2]

Explanation:
Conditions 1 and 2 directly contradict each other because they require opposite truth values for Alice's decision.

[UNSAT]
"""


def build_no_leak_prompt_from_record(rec: Dict[str, Any]) -> Optional[str]:
    """
    Rebuild the RL prompt from the record's user_prompt, but remove:
      - <satisfiable>
      - <label>
      - Z3 certificate
      - unsat core
      - sat assignment

    We keep:
      - scenario
      - variable_mapping
      - conditions
      - question
      - dims
      - num_vars
      - num_clauses
      - clauses_dimacs
      - readable_cnf
    """
    user_prompt = rec.get("user_prompt", "")
    if not user_prompt:
        return None

    scenario = extract_tag(user_prompt, "scenario")
    variable_mapping = extract_tag(user_prompt, "variable_mapping")
    conditions = extract_tag(user_prompt, "conditions")
    question = extract_tag(user_prompt, "question")

    dims = extract_tag(user_prompt, "dims")
    num_vars = extract_tag(user_prompt, "num_vars")
    num_clauses = extract_tag(user_prompt, "num_clauses")
    clauses_dimacs = extract_tag(user_prompt, "clauses_dimacs")
    readable_cnf = extract_tag(user_prompt, "readable_cnf")

    required = [scenario, variable_mapping, conditions, question, dims, num_vars, num_clauses, clauses_dimacs, readable_cnf]
    if any(x is None for x in required):
        return None

    user = f"""Now solve the actual puzzle below. Do not copy the examples. Use the same output format.
    
## Puzzle

<scenario>
{scenario}
</scenario>

<variable_mapping>
{variable_mapping}
</variable_mapping>

<conditions>
{conditions}
</conditions>

<question>
{question}
</question>

## Formal SAT information

<dims>
{dims}
</dims>

<num_vars>
{num_vars}
</num_vars>

<num_clauses>
{num_clauses}
</num_clauses>

<clauses_dimacs>
{clauses_dimacs}
</clauses_dimacs>

<readable_cnf>
{readable_cnf}
</readable_cnf>

## Instruction

Solve the puzzle. Produce a SAT/UNSAT decision and a verifiable certificate in the required format.
"""

    return f"{SAFE_SYSTEM_PROMPT}\n\n{user}\n\nAssistant:\n"


def condition_to_raw_clause_map(gt: Dict[str, Any]) -> Dict[str, int]:
    """
    Build mapping from 1-indexed condition numbers to raw 0-indexed clause indices.
    Example:
      condition 4 -> raw clause index 0
    """
    out = {}
    for item in gt.get("mapping_details", []) or []:
        raw_idx = item.get("raw_clause_index")
        for cnum in item.get("condition_numbers", []) or []:
            out[str(int(cnum))] = int(raw_idx)
    return out


def get_used_prompt_ids() -> set:
    """
    Optional: exclude examples already used in SFT train/test splits if those files
    include prompt_id. If not, this safely returns a smaller/empty set.
    """
    used = set()
    paths = [
        Path("dataset/sft_by_label/sat/train.jsonl"),
        Path("dataset/sft_by_label/sat/test.jsonl"),
        Path("dataset/sft_by_label/unsat/train.jsonl"),
        Path("dataset/sft_by_label/unsat/test.jsonl"),
    ]
    for p in paths:
        for row in load_jsonl(p):
            for key in ["prompt_id", "dataset_id", "row_id"]:
                if row.get(key) is not None:
                    used.add(str(row[key]))
    return used


used_prompt_ids = get_used_prompt_ids()
print("Used prompt ids found from SFT splits:", len(used_prompt_ids))

Used prompt ids found from SFT splits: 200


In [6]:
# ============================================================
# Cell 3: load unused records and create GRPO dataset
# ============================================================

record_paths = sorted(RECORD_DIR.glob("row_*.json"))
print("Found record files:", len(record_paths))

records = []
for p in record_paths:
    try:
        rec = load_json(p)
    except Exception as e:
        print("Skipping bad JSON:", p, e)
        continue

    # Keep only teacher outputs that were correct and not truncated.
    if rec.get("teacher_label_matches_ground_truth") is not True:
        continue

    if rec.get("generation_info", {}).get("likely_truncated_by_max_new_tokens") is True:
        continue

    prompt_id = str(rec.get("prompt_id", ""))
    if prompt_id and prompt_id in used_prompt_ids:
        continue

    gt = rec.get("ground_truth") or {}
    label = (gt.get("label") or rec.get("ground_truth_label") or rec.get("label") or "").upper()

    if label not in {"SAT", "UNSAT"}:
        continue

    prompt = build_no_leak_prompt_from_record(rec)
    if prompt is None:
        continue

    clauses = gt.get("clauses")
    if clauses is None:
        # Fallback: parse from user prompt
        clauses = parse_tag_json(rec.get("user_prompt", ""), "clauses_dimacs")

    dims = gt.get("dims")
    if dims is None:
        dims = parse_tag_json(rec.get("user_prompt", ""), "dims")

    num_vars = gt.get("num_vars")
    if num_vars is None:
        num_vars = int(extract_tag(rec.get("user_prompt", ""), "num_vars"))

    if clauses is None or dims is None or num_vars is None:
        continue

    records.append({
        "prompt": prompt,
        "prompt_id": prompt_id,
        "row_index": int(rec.get("row_index_in_prompt_file", rec.get("index", -1))),
        "label": label,
        "dims_json": json.dumps(dims),
        "num_vars": int(num_vars),
        "clauses_json": json.dumps(clauses),
        "condition_to_raw_json": json.dumps(condition_to_raw_clause_map(gt)),
    })

print("Good unused records:", len(records))

sat_rows = [r for r in records if r["label"] == "SAT"]
unsat_rows = [r for r in records if r["label"] == "UNSAT"]

random.shuffle(sat_rows)
random.shuffle(unsat_rows)

selected = sat_rows[:N_PER_LABEL] + unsat_rows[:N_PER_LABEL]
random.shuffle(selected)

print("Selected SAT:", sum(r["label"] == "SAT" for r in selected))
print("Selected UNSAT:", sum(r["label"] == "UNSAT" for r in selected))
print("Selected total:", len(selected))

if len(selected) == 0:
    raise ValueError("No RL training examples selected. Check RECORD_DIR and filters.")

train_dataset = Dataset.from_list(selected)

print(train_dataset)
print(train_dataset[0].keys())
print("Example prompt preview:")
print(train_dataset[0]["prompt"][:2000])

Found record files: 2087
Good unused records: 1256
Selected SAT: 8
Selected UNSAT: 8
Selected total: 16
Dataset({
    features: ['prompt', 'prompt_id', 'row_index', 'label', 'dims_json', 'num_vars', 'clauses_json', 'condition_to_raw_json'],
    num_rows: 16
})
dict_keys(['prompt', 'prompt_id', 'row_index', 'label', 'dims_json', 'num_vars', 'clauses_json', 'condition_to_raw_json'])
Example prompt preview:
You are a logical reasoning assistant solving SATBench-style natural-language logic puzzles.

Important rules:
- Use only the constraints stated in the <conditions> section.
- The <scenario> section is background only and adds no hidden constraints.
- All variables are independent Boolean decisions unless the conditions explicitly say otherwise.
- Do not add commonsense assumptions such as mutual exclusivity, exactly-one constraints, or real-world causal links.
- Variables not mentioned in the conditions are irrelevant to satisfiability.

Required output format:

<think>
Write concise 

In [7]:
# ============================================================
# Cell 4: Z3 certificate validation helpers for rewards
# ============================================================

def completion_to_text(completion) -> str:
    """
    TRL can pass completions as strings for standard format,
    or as message dictionaries for conversational format.
    """
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        if len(completion) > 0 and isinstance(completion[0], dict):
            return completion[0].get("content", "")
    return str(completion)


def extract_final_label(text: str) -> Optional[str]:
    # Prefer final bracket label.
    labels = re.findall(r"\[(SAT|UNSAT)\]", text, flags=re.IGNORECASE)
    if labels:
        return labels[-1].upper()

    # Fallback to Decision or Label line.
    m = re.search(r"\b(?:Decision|Label)\s*:\s*(SAT|UNSAT)\b", text, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()

    return None


def extract_balanced_bracket(text: str, start_idx: int) -> Optional[str]:
    i = text.find("[", start_idx)
    if i < 0:
        return None

    depth = 0
    for j in range(i, len(text)):
        if text[j] == "[":
            depth += 1
        elif text[j] == "]":
            depth -= 1
            if depth == 0:
                return text[i:j + 1]
    return None


def flatten_assignment(obj) -> List[bool]:
    flat = []

    def rec(x):
        if isinstance(x, list):
            for y in x:
                rec(y)
        elif isinstance(x, bool):
            flat.append(bool(x))
        elif isinstance(x, int):
            flat.append(bool(x))
        elif isinstance(x, str):
            t = x.strip().lower()
            if t in {"1", "true", "t", "yes"}:
                flat.append(True)
            elif t in {"0", "false", "f", "no"}:
                flat.append(False)

    rec(obj)
    return flat


def extract_assignment(text: str, num_vars: int) -> Dict[int, bool]:
    """
    Preferred format:
      Assignment: [[0, 1], [1, 0], ...]
    Returns DIMACS-style 1-indexed variable assignment:
      {1: False, 2: True, ...}
    """
    for m in re.finditer(r"Assignment\s*:", text, flags=re.IGNORECASE):
        block = extract_balanced_bracket(text, m.end())
        if not block:
            continue

        try:
            obj = ast.literal_eval(block)
        except Exception:
            try:
                obj = json.loads(block)
            except Exception:
                continue

        flat = flatten_assignment(obj)
        if flat:
            return {i + 1: bool(v) for i, v in enumerate(flat[:num_vars])}

    return {}


def extract_unsat_core_condition_numbers(text: str) -> List[int]:
    """
    Preferred format:
      UNSAT core condition numbers: [1, 3, 4]
    """
    patterns = [
        r"UNSAT\s+core\s+condition\s+numbers\s*:\s*\[([^\]]+)\]",
        r"UNSAT\s+core\s+conditions?\s*:\s*\[([^\]]+)\]",
        r"core\s+condition\s+numbers\s*:\s*\[([^\]]+)\]",
        r"conditions?\s*:\s*\[([^\]]+)\]",
    ]

    for pat in patterns:
        m = re.search(pat, text, flags=re.IGNORECASE)
        if m:
            nums = [int(x) for x in re.findall(r"\d+", m.group(1))]
            if nums:
                return nums

    # Fallback for prose like "conditions 1, 3, and 4"
    m = re.search(r"conditions?\s+((?:\d+\s*(?:,|and|&)?\s*){1,20})", text, flags=re.IGNORECASE)
    if m:
        nums = [int(x) for x in re.findall(r"\d+", m.group(1))]
        if nums:
            return nums

    return []


def z3_vars(num_vars: int):
    return {i: Bool(f"x{i}") for i in range(1, num_vars + 1)}


def z3_clause_expr(clause: List[int], vars_by_id):
    exprs = []
    for lit in clause:
        v = vars_by_id[abs(int(lit))]
        exprs.append(v if lit > 0 else Not(v))
    return Or(*exprs)


def check_assignment_satisfies_all_clauses(clauses: List[List[int]], num_vars: int, assignment: Dict[int, bool]) -> bool:
    if not assignment:
        return False

    for clause in clauses:
        clause_ok = False

        for lit in clause:
            var_id = abs(int(lit))
            if var_id not in assignment:
                continue

            value = assignment[var_id]
            literal_value = value if lit > 0 else (not value)

            if literal_value:
                clause_ok = True
                break

        if not clause_ok:
            return False

    return True


def check_unsat_core_with_z3(clauses: List[List[int]], num_vars: int, raw_clause_indices: List[int]) -> bool:
    if not raw_clause_indices:
        return False

    if any(i < 0 or i >= len(clauses) for i in raw_clause_indices):
        return False

    vars_by_id = z3_vars(num_vars)
    solver = Solver()

    for idx in raw_clause_indices:
        solver.add(z3_clause_expr(clauses[idx], vars_by_id))

    return solver.check() == unsat

In [8]:
# ============================================================
# Cell 5: GRPO reward functions
# ============================================================

def format_reward_func(completions, **kwargs):
    """
    Small reward for following the required output format.
    """
    rewards = []

    for completion in completions:
        text = completion_to_text(completion).strip()
        r = 0.0

        if re.search(r"\[(SAT|UNSAT)\]\s*$", text, flags=re.IGNORECASE):
            r += 0.3

        if re.search(r"\bDecision\s*:\s*(SAT|UNSAT)\b", text, flags=re.IGNORECASE):
            r += 0.2

        if re.search(r"\bCertificate\s*:", text, flags=re.IGNORECASE):
            r += 0.2

        if re.search(r"\bAssignment\s*:", text, flags=re.IGNORECASE) or re.search(r"\bUNSAT\s+core\b", text, flags=re.IGNORECASE):
            r += 0.3

        rewards.append(r)

    return rewards


def label_reward_func(completions, label, **kwargs):
    """
    Reward correct SAT/UNSAT classification.
    """
    rewards = []

    for completion, gold in zip(completions, label):
        text = completion_to_text(completion)
        pred = extract_final_label(text)
        rewards.append(1.0 if pred == gold else -0.5)

    return rewards


def z3_certificate_reward_func(completions, label, clauses_json, num_vars, condition_to_raw_json, **kwargs):
    """
    Reward solver-verifiable certificates.

    SAT:
      Reward if the generated assignment satisfies every CNF clause.

    UNSAT:
      Reward if the generated UNSAT core condition numbers map to a subset
      of clauses that Z3 proves UNSAT.
    """
    rewards = []

    for completion, gold, clauses_s, nvars, cond_map_s in zip(
        completions, label, clauses_json, num_vars, condition_to_raw_json
    ):
        text = completion_to_text(completion)
        pred = extract_final_label(text)

        clauses = json.loads(clauses_s)
        nvars = int(nvars)
        condition_to_raw = json.loads(cond_map_s) if cond_map_s else {}

        # If the final SAT/UNSAT label is wrong, no certificate reward.
        if pred != gold:
            rewards.append(0.0)
            continue

        if gold == "SAT":
            assignment = extract_assignment(text, nvars)
            valid = check_assignment_satisfies_all_clauses(clauses, nvars, assignment)
            rewards.append(2.0 if valid else 0.0)

        elif gold == "UNSAT":
            condition_numbers = extract_unsat_core_condition_numbers(text)

            raw_indices = []
            for cnum in condition_numbers:
                # Prefer explicit condition-to-raw mapping from ground_truth.
                if str(cnum) in condition_to_raw:
                    raw_indices.append(int(condition_to_raw[str(cnum)]))
                else:
                    # Fallback: assume condition i maps to raw clause i-1.
                    raw_indices.append(int(cnum) - 1)

            valid = check_unsat_core_with_z3(clauses, nvars, sorted(set(raw_indices)))
            rewards.append(2.0 if valid else 0.0)

        else:
            rewards.append(0.0)

    return rewards


# Quick sanity check with fake completions
fake_completions = [
    """
<think>...</think>
Decision: UNSAT
Certificate:
UNSAT core condition numbers: [1, 2, 3, 4]
Explanation:
The listed conditions are contradictory.
[UNSAT]
"""
]

sample = train_dataset[0]
print("Sample label:", sample["label"])
print("Format reward:", format_reward_func(fake_completions))
print("Label reward:", label_reward_func(fake_completions, [sample["label"]]))
print(
    "Z3 reward:",
    z3_certificate_reward_func(
        fake_completions,
        [sample["label"]],
        [sample["clauses_json"]],
        [sample["num_vars"]],
        [sample["condition_to_raw_json"]],
    ),
)

Sample label: UNSAT
Format reward: [1.0]
Label reward: [1.0]
Z3 reward: [0.0]


In [9]:
# ============================================================
# Cell 6: tokenizer and GRPOConfig
# ============================================================

if not MODEL_DIR.exists():
    raise FileNotFoundError(f"Cannot find SFT model directory: {MODEL_DIR}")

tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    trust_remote_code=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# TRL recommends left padding for generation-style training.
tokenizer.padding_side = "left"

print("pad_token:", tokenizer.pad_token, tokenizer.pad_token_id)
print("eos_token:", tokenizer.eos_token, tokenizer.eos_token_id)


def make_grpo_config(**kwargs):
    """
    Makes the notebook more robust across slightly different TRL versions.
    Unsupported kwargs are dropped with a printout.
    """
    valid = set(inspect.signature(GRPOConfig.__init__).parameters.keys())
    filtered = {}
    dropped = {}

    for k, v in kwargs.items():
        if k in valid:
            filtered[k] = v
        else:
            dropped[k] = v

    if dropped:
        print("Dropped unsupported GRPOConfig keys:")
        for k in dropped:
            print("  -", k)

    return GRPOConfig(**filtered)


use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = make_grpo_config(
    output_dir=str(OUTPUT_DIR),

    # Tiny notebook run.
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,

    # Important: effective batch size must be divisible by num_generations.
    num_generations=NUM_GENERATIONS,

    # Generation length.
    max_prompt_length=MAX_PROMPT_LENGTH,
    max_completion_length=MAX_COMPLETION_LENGTH,
    temperature=0.7,
    top_p=0.95,

    # Small LR because this is RL after SFT.
    learning_rate=5e-6,
    warmup_ratio=0.0,
    weight_decay=0.0,
    max_grad_norm=1.0,

    # KL coefficient. Setting beta > 0 uses a reference model.
    # For quick tests, 0.001 is a conservative value.
    beta=0.001,

    # Memory.
    gradient_checkpointing=True,
    bf16=use_bf16,
    fp16=use_fp16,

    # Keep hidden columns for reward functions.
    remove_unused_columns=False,

    # Logging/saving.
    logging_steps=1,
    logging_first_step=True,
    save_steps=MAX_STEPS,
    save_total_limit=2,
    report_to="none",

    # Do not use vLLM in this notebook test.
    use_vllm=False,

    # Load model from local folder with correct dtype.
    model_init_kwargs={
        "trust_remote_code": True,
        "torch_dtype": "bfloat16" if use_bf16 else ("float16" if use_fp16 else "float32"),
    },

    seed=SEED,
)

print(training_args)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


pad_token: <|endoftext|> 248044
eos_token: <|endoftext|> 248044
Dropped unsupported GRPOConfig keys:
  - max_prompt_length
GRPOConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
beta=0.001,
bf16=True,
bf16_full_eval=False,
cache_implementation=None,
cast_lm_head_to_fp32=False,
chat_template_kwargs=None,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
delta=None,
disable_dropout=False,
disabl

In [10]:
# ============================================================
# Cell 7: run small GRPO + Z3 reward training
# ============================================================

trainer = GRPOTrainer(
    model=str(MODEL_DIR),
    args=training_args,
    reward_funcs=[
        format_reward_func,
        label_reward_func,
        z3_certificate_reward_func,
    ],
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

train_result = trainer.train()

print("Train result:")
print(train_result)

# Save this tiny exploratory GRPO model.
final_dir = OUTPUT_DIR / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

print("Saved GRPO test model to:", final_dir)

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Step,Training Loss
1,0.000000
2,0.000000
3,0.000005
4,0.000004
5,0.000012


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Train result:
TrainOutput(global_step=5, training_loss=4.192056371721264e-06, metrics={'train_runtime': 518.7898, 'train_samples_per_second': 0.019, 'train_steps_per_second': 0.01, 'total_flos': 0.0, 'train_loss': 4.192056371721264e-06})


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved GRPO test model to: results/grpo_z3_qwen35_08b_test/final_model
